# Irene neural avatar test (free Google Colab)

This notebook runs the official LivePortrait human model on a temporary Colab GPU. It uploads a source photo, animates it with a short driving clip, previews the result, and lets you download the MP4.

Colab sessions are temporary and free GPU access is not guaranteed. Do not upload anything you do not want processed by Google Colab.

In [1]:
import os
import shutil
import torch

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU was assigned. In Colab choose Runtime > Change runtime type > T4 GPU, then rerun this cell.')
print('Free disk (GB):', round(shutil.disk_usage('/content').free / 1024**3, 1))

GPU available: True
GPU: Tesla T4
Free disk (GB): 65.9


In [ ]:
!git clone -q --depth 1 https://github.com/KlingTeam/LivePortrait.git /content/LivePortrait
%cd /content/LivePortrait
!pip install -q -r requirements.txt
!pip install -q "huggingface_hub[cli]"

In [ ]:
%cd /content/LivePortrait
!huggingface-cli download KlingTeam/LivePortrait --local-dir pretrained_weights --exclude "*.git*" "README.md" "docs"
print('Pretrained weights are ready.')

In [ ]:
from google.colab import files

print('Choose Irene photo (PNG or JPG).')
uploaded = files.upload()
source_path = '/content/' + next(iter(uploaded))
print('Source photo:', source_path)

In [ ]:
%cd /content/LivePortrait
!python inference.py -s "{source_path}" -d assets/examples/driving/d0.mp4 -o animations --flag_crop_driving_video --driving_option expression-friendly

In [ ]:
from glob import glob
from IPython.display import Video, display

outputs = sorted(glob('/content/LivePortrait/animations/*.mp4'))
if not outputs:
    raise FileNotFoundError('LivePortrait did not produce an MP4 file.')
output_path = outputs[-1]
print('Result:', output_path)
display(Video(output_path, embed=True, width=512))

In [ ]:
from google.colab import files
files.download(output_path)